In [1]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm


In [2]:
sys.path.insert(0, "/lustre/lrspec/users/4301/ABC-SN/code")
from data_degrading import degrade_spectrum
import abcsn_training
import abcsn_config

sys.path.insert(0, "/lustre/lrspec/users/4301/Milligan_project")
from process_data import *


sys.path.insert(0, "/lustre/lrspec/users/4301/snidpy/sourcepy")
from apodize import *
from logwave import Logwave as lw
from logwave import log_rebin

2026-01-29 15:06:27.981796: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-29 15:06:28.036682: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-29 15:06:30.257316: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [23]:
abcsn = keras.models.load_model("/lustre/lrspec/users/4301/ABC-SN/abcsn/ABCSN.keras", compile=False)

In [19]:
# used:
# abcsn_config.SN_Stypes_int_to_str replaced with ABC_subtype_id_to_str
# to make the dictionary corresponding to the labels used above
# may just want to change the above to use the same strings as int_to_str function
ABC_subtype_id_to_str = {
    0: "Ia-norm",
    1: "Ia-91T",
    2: "Ia-91bg",
    3: "Iax",
    4: "Ib-norm",
    5: "Ibn",
    6: "IIb",
    7: "Ic-norm",
    8: "Ic-broad",
    9: "IIP",
}

ABC_ID_dict ={"Ia": 0,
          "Iap": 0,
          "Ic": 7,
          "Ib": 4,
          "II": 9, # IIP = type 2 plateau = normal
          "IIb": 6,
          "IIn": 9,
          "SL": None,
          "TDE": None,
          "CaRT": None
          }

Mill_ID_dict ={"Ia": 0,
          "Iap": 0,
          "Ic": 1,
          "Ib": 1,
          "II": 2, # IIP = type 2 plateau = normal
          "IIb": 1,
          "IIn": 2,
          "SL": 3,
          "TDE": 4,
          "CaRT": 4
          }

# five types recorded in Milligan et al.
Mill_types_to_int = {0: "Ia",
                     1: "Ib & Ic",
                     2: "II",
                     3: "SLSN",
                     4: "Non-SN",
                     5: "other"
                     }

# convert ABC types to Mill categories
ABC_to_Mill ={0:0,   # Ia-norm -> Ia
              1:0,   # Ia-91T -> Ia
              2:0,   # Ia-91bg -> Ia
              3:0,   # Iax -> Ia
              4:1,   # Ib-norm -> Ib & Ic
              5:1,   # Ibn -> Ib & Ic
              6:1,   # IIb -> Ib & Ic
              7:1,  # Ic-norm -> Ib & Ic
              8:1,  # Ic-broad -> Ib & Ic
              9:2,  # IIP -> II
            }


In [28]:
folder_list = ["kinney_sample1"]

for folder in folder_list:
    filenames = sorted(glob.glob("/lustre/lrspec/users/4301/Milligan_ABC-SN/data/"+ folder +"/*"))
    file_info = [name.split("/")[-1] for name in filename]
    df_metadata = pd.DataFrame(file_info, columns=["filename"])
    df_metadata[["host", "sn_type", "redshift", "SN_mag", "host_mag"]] = [get_filename_info(info) for info in file_info]
    df_metadata[ "host_mag"] = df_metadata[ "host_mag"].astype(float)
    df_metadata[ "redshift"] = df_metadata[ "redshift"].astype(float)
    df_metadata[ "SN_mag"] = df_metadata[ "SN_mag"].astype(float)

    # set up to extract the X values 
    num_wvl = 139
    dat_size = len(df_metadata)
    plots = False
    
    X_all = np.zeros((dat_size, 1, num_wvl))
    Y_ABC_IDs = np.zeros((dat_size)) # classification as the float classifier
    Y_Mill_IDs = np.zeros((dat_size)) # classification as the float classifier
    
    for i in range(dat_size):
      wvl, X = process_files(filename[i], 4000, 7000, plot_spectra = plots, verbose=False)
      X_all[i] = X
      # dont save wvl as all are the same
    
      try:
        Y_ABC_IDs[i] = ABC_ID_dict[df_metadata.sn_type[i]]
        Y_Mill_IDs[i] = Mill_ID_dict[df_metadata.sn_type[i]]
    
      except Exception as e:
        print(df_metadata.sn_type[i], i)
        raise e
    df_metadata["ABC_ID"] = Y_ABC_IDs
    df_metadata["Mill_ID"] = Y_Mill_IDs

    # predict values
    X = X_all.copy()
    P = abcsn.predict(X, verbose=0)
    P_argmax = np.argmax(P, axis=1)
    df_metadata["Pred_ABC_ID"] = P_argmax
    df_metadata["Pred_Mill_ID"] = np.array([ABC_to_Mill[i] for i in P_argmax])
    df_metadata.to_csv(
        path_or_buf="/lustre/lrspec/users/4301/Milligan_ABC-SN/data/csvs/" + folder + ".csv", 
        index=False, 
        lineterminator='\n')


meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
meanzero
m

2026-01-29 15:45:48.685522: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


In [29]:
df_metadata.head()

,filename,host,sn_type,redshift,SN_mag,host_mag,ABC_ID,Mill_ID,Pred_ABC_ID,Pred_Mill_ID
0,ellipticalCRTSmag22.627Gmag21.98z0.14306texp62...,elliptical,CaRT,0.14306,22.627,21.980,NaN,4.0,3,0
1,ellipticalIa_Smag19.556Gmag22.765z0.09895texp6...,elliptical,Ia,0.09895,19.556,22.765,0.0,0.0,0,0
2,ellipticalIa_Smag19.834Gmag22.761z0.24084texp5...,elliptical,Ia,0.24084,19.834,22.761,0.0,0.0,0,0
3,ellipticalIa_Smag20.338Gmag21.747z0.19219texp1...,elliptical,Ia,0.19219,20.338,21.747,0.0,0.0,3,0
4,ellipticalIa_Smag20.428Gmag20.809z0.1662texp19...,elliptical,Ia,0.16620,20.428,20.809,0.0,0.0,3,0
